## Section 2 — Configuration & Initialization
Load the master config, resolve device, and validate all pipeline parameters before touching any data.

In [ ]:
# config.yaml is the single source of truth for ALL parameters.
# Never hardcode dataset names, batch sizes, or class lists here.

from src.utils import load_config, setup_logging, set_seed, get_device
from src.utils import get_per_class_target, get_candidate_pool_size, get_pipeline_mode

# Load config from the cloned repo
config = load_config('configs/config.yaml')

# Initialize logging (writes to logs/pipeline.log + console)
setup_logging(config)

# Set all random seeds for reproducibility
# This covers: Python random, NumPy, PyTorch CPU, PyTorch CUDA
set_seed(config['dataset']['seed'])

print('Config loaded successfully')
print(f"Pipeline mode   : {get_pipeline_mode(config)}")
print(f"Dataset         : {config['dataset']['name']}")
print(f"Split           : {config['dataset']['split']}")
print(f"Target N        : {config['dataset']['target_n']}")
print(f"Classes ({len(config['dataset']['classes'])})     : {config['dataset']['classes']}")
print(f"Per-class target: {get_per_class_target(config)}")
print(f"Candidate pool  : {get_candidate_pool_size(config)} (= target × {config['dataset']['pool_multiplier']}x)")
print(f"Seed            : {config['dataset']['seed']}")

In [ ]:
# utils.get_device() checks CUDA → MPS → CPU in order.
# On Colab Pro with A100, this always returns cuda.

import torch
from src.utils import get_device, get_pin_memory

device = get_device()

print(f'Device          : {device}')
print(f'Pin memory      : {get_pin_memory(device, config)}')
print(f'Batch size      : {config["hardware"]["batch_size"]}')

if device.type == 'cuda':
    # Log available VRAM before any models are loaded
    free_vram  = torch.cuda.mem_get_info(0)[0] / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM free       : {free_vram:.1f} / {total_vram:.1f} GB')

# Guard: batch_size > 64 risks OOM on A100 for CLIP+GPT-2 simultaneously
# Reduce to 32 in config.yaml if you see CUDA OOM errors
assert config['hardware']['batch_size'] <= 64, (
    f"batch_size={config['hardware']['batch_size']} may OOM. "
    "Reduce to 32 or less in config.yaml."
)

In [ ]:
# Quick dry-run: verify HuggingFace Hub can find both model configs
# before we start downloading COCO data.
# This fails fast if model names are wrong or Hub is unreachable.

from transformers import AutoConfig

clip_name = config['models']['clip']
gpt2_name = config['models']['gpt2']

print(f'Checking CLIP model  : {clip_name}')
clip_cfg = AutoConfig.from_pretrained(clip_name)
print(f'  → model_type       : {clip_cfg.model_type}')
print(f'  → projection_dim   : {clip_cfg.projection_dim}')

print(f'Checking GPT-2 model : {gpt2_name}')
gpt2_cfg = AutoConfig.from_pretrained(gpt2_name)
print(f'  → model_type       : {gpt2_cfg.model_type}')
print(f'  → hidden_size      : {gpt2_cfg.n_embd}')
print(f'  → vocab_size       : {gpt2_cfg.vocab_size}')

# Store dims for use in later sections
CLIP_EMBED_DIM = clip_cfg.projection_dim  # 512 for ViT-B/32
GPT2_HIDDEN_DIM = gpt2_cfg.n_embd         # 768 for gpt2-base

print(f'\nEmbedding dimensions confirmed:')
print(f'  CLIP image embedding : {CLIP_EMBED_DIM}')
print(f'  GPT-2 hidden size    : {GPT2_HIDDEN_DIM}')
print('\n Model configs resolved — safe to proceed')

In [ ]:

# Ensure outputs/ logs/ and checkpoints/ exist before the pipeline runs.
# These are .gitignored but must exist at runtime.

from pathlib import Path
from src.utils import get_output_path

# Resolve output path from config (creates outputs/ if it doesn't exist)
output_path = get_output_path(config)

# Create logs/ directory
log_dir = Path(config['logging']['log_dir'])
log_dir.mkdir(parents=True, exist_ok=True)

# Create checkpoints/ directory for M3
Path('checkpoints').mkdir(exist_ok=True)

print(f'Output path   : {output_path}')
print(f'Log dir       : {log_dir}')
print(f'Checkpoint dir: checkpoints/')
print('\n All output directories ready')